#### PubMLP: Automatic Publication Classifier Using MLP

In [ ]:
import pandas as pd
import numpy as np
import spacy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from transformers import BertTokenizer, BertConfig, BertModel
from torch.utils.data.sampler import RandomSampler, SequentialSampler
import matplotlib.pyplot as plt
import warnings

#### Load


In [ ]:
warnings.filterwarnings('ignore',category=FutureWarning)
# Load the initial data
data_init = pd.read_csv('../files/data.csv')

extract_country = spacy.load('en_core_web_sm')

# Define a function to extract country entities using NER
def extract_country_entities(text):
    # Load the spaCy model (English)
    if pd.notna(text):  # Check if the text is not NaN
        doc = extract_country(text)
        country = [ent.text for ent in doc.ents if ent.label_ == 'GPE']
        return ', '.join(country)
    else:
        return ''

# Apply the function to the 'C1' column and create a new 'Country' column
data_init['Country'] = data_init['C1'].apply(extract_country_entities)

# Select relevant columns
raw_df = data_init[['UT', 'included', 'single_case', 'technology_use',
                'PY', 'AF', 'TI', 'AB', 'DE', 'ID', 'SO', 'CR', 'Country']].iloc[:, :]

# Convert PY (publication year) to numeric format
raw_df['PY'] = pd.to_numeric(raw_df['PY'], errors='coerce')

# Check columns with text data
raw_df.describe(include=object)

# Check columns with numeric data
raw_df.describe()

missing_values = raw_df.isnull().sum()
missing_values

df = raw_df.copy()

# Replace missing values
df.dropna(subset=['PY'], inplace=True)

# Split the data for training, validation, and testing purposes
train_df, valid_df, test_df = np.split(
    df.sample(frac=1, random_state=42), [
        int(0.8 * len(df)), int(0.9 * len(df))
    ]
)

print(train_df.head(5))
print(f"Training data number: {len(train_df)}")
print(f"Validation data number: {len(valid_df)}")
print(f"Testing data number: {len(test_df)}")

# Save data to CSV files
train_df.to_excel('../files/train_data.xlsx', index=False)
valid_df.to_excel('../files/valid_data.xlsx', index=False)
test_df.to_excel('../files/test_data.xlsx', index=False)

#### Preprocess 


In [ ]:
# Set up settings for reproducible results
torch.backends.cudnn.deterministic = True
random_seed = 2023
torch.manual_seed(random_seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Preprocess text, categorical, and numeric Data
def preprocess_dataset(data, tokenizer, device):
    text_cols = data[['AF', 'TI', 'AB', 'DE', 'ID', 'SO', 'CR', 'Country']]
    text_embeddings = []
    attention_masks = []
    for (i, row) in data.iterrows():  
        combined = ''
        for col in text_cols:  
            combined += (str(row[col]) + '[SEP] ')
        tokenized = tokenizer.encode_plus(
            combined,
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        text_embeddings.append(tokenized["input_ids"])
        attention_masks.append(tokenized["attention_mask"])
    
    text_embeddings = torch.cat(text_embeddings, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)

    # Convert categorical data (one-hot encoding)
    categorical_cols = data[['single_case', 'technology_use']]
    encoder = OneHotEncoder(handle_unknown='ignore')
    encoded_categorical_cols = encoder.fit_transform(categorical_cols)
    encoded_categorical_cols_np = encoded_categorical_cols.toarray()
    categorical_tensor = torch.tensor(encoded_categorical_cols_np, dtype=torch.float32).to(device)

    # Convert numerical data (publication year) 
    numerical_cols = data[['PY']]
    numerical_cols_np = numerical_cols.values
    numeric_tensor = torch.tensor(numerical_cols_np, dtype=torch.float32).to(device)

    # Concatenate all the input data tensors
    combined_tensors = torch.cat([text_embeddings, attention_masks, categorical_tensor, numeric_tensor], dim=1)
    input_size = combined_tensors.size(1)

    # Convert the label data (one-hot encoding)
    label_col = data[['included']]
    encoder = OneHotEncoder(handle_unknown='ignore')
    encoded_label_col = encoder.fit_transform(label_col)
    encoded_label_col_np = encoded_label_col.toarray()
    label_tensor = torch.tensor(encoded_label_col_np, dtype=torch.float32).to(device)

    return TensorDataset(combined_tensors, label_tensor)  

# Create a DataLoader
def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader
    
# Load the BERT tokenizer
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

train_dataset = preprocess_dataset(train_df, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size = 32)

valid_dataset = preprocess_dataset(valid_df, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size = 32)

test_dataset = preprocess_dataset(test_df, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size = 32)

print(train_dataset[0])

#### Train PubMLP


In [ ]:
# Define the multilayer perceptron model 
class PubMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout_prob=0.5):
        super(PubMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_prob)  
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)  
        x = self.fc2(x)
        return x

# Define hyperparameters
input_size = 1029
hidden_size = 32  
num_classes = 2

model = PubMLP(input_size, hidden_size, num_classes).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  
optimizer = optim.Adam(model.parameters(), lr=0.001) 

epochs = 4

training_losses = []
validation_losses = []
test_losses = []
training_accuracies = []
validation_accuracies = []
test_accuracies = []

# Evaluation on the training dataset
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    total_correct = 0

    for batch in train_dataloader:
        inputs, labels = batch
        inputs = inputs.to(device)
        labels = labels.argmax(dim=1).long().to(device)  

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_correct += (outputs.argmax(dim=1) == labels).sum().item()

    average_loss = total_loss / len(train_dataloader)
    accuracy = total_correct / len(train_dataset)

    print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {average_loss:.4f}, Training Accuracy: {accuracy:.4f}")
    
    # Evaluation on the validation dataset
    model.eval()
    valid_loss = 0.0
    valid_correct = 0

    with torch.no_grad():
        for batch in valid_dataloader:
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.argmax(dim=1).long().to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            valid_loss += loss.item()
            valid_correct += (outputs.argmax(dim=1) == labels).sum().item()

    valid_average_loss = valid_loss / len(valid_dataloader)
    valid_accuracy = valid_correct / len(valid_dataset)

    print(f"Validation Loss: {valid_average_loss:.4f}, Validation Accuracy: {valid_accuracy:.4f}")

    # Evaluation on the test dataset
    test_loss = 0.0
    test_correct = 0

    with torch.no_grad():
        for batch in test_dataloader:
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.argmax(dim=1).long().to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            test_correct += (outputs.argmax(dim=1) == labels).sum().item()

    test_average_loss = test_loss / len(test_dataloader)
    test_accuracy = test_correct / len(test_dataset)

    print(f"Test Loss: {test_average_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

    training_losses.append(average_loss)
    validation_losses.append(valid_average_loss)
    test_losses.append(test_average_loss)
    training_accuracies.append(accuracy)
    validation_accuracies.append(valid_accuracy)
    test_accuracies.append(test_accuracy)

#### Plot 


In [ ]:
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), training_losses, label='Training Loss', marker='o')
plt.plot(range(1, epochs + 1), validation_losses, label='Validation Loss', marker='o')
plt.plot(range(1, epochs + 1), test_losses, label='Test Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training, Validation, and Test Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), training_accuracies, label='Training Accuracy', marker='o')
plt.plot(range(1, epochs + 1), validation_accuracies, label='Validation Accuracy', marker='o')
plt.plot(range(1, epochs + 1), test_accuracies, label='Test Accuracy', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training, Validation, and Test Accuracy')
plt.legend()

plt.tight_layout()
plt.show()